In [ ]:
import sys, os, re

import qubx
from qubx import logger

%qubxd

%load_ext autoreload
%autoreload 2

from qubx.core.basics import Signal, TimestampedDict, DataType
from qubx.core.lookups import lookup
from qubx.core.interfaces import IStrategy, IStrategyContext, TriggerEvent, MarketEvent, IStrategyInitializer
from qubx.data.registry import StorageRegistry
from qubx.data.transformers import TypedGenericSeries

from qubx.backtester.simulator import simulate
from qubx.utils.time import handle_start_stop


⠀⠀⡰⡖⠒⠒⢒⢦⠀⠀   
⠀⢠⠃⠈⢆⣀⣎⣀⣱⡀  QUBX | Quantitative Backtesting Environment 
⠀⢳⠒⠒⡞⠚⡄⠀⡰⠁         (c) 2025, ver. 0.7.40.dev8
⠀⠀⠱⣜⣀⣀⣈⣦⠃⠀⠀⠀ 
        


In [2]:
stor = StorageRegistry.get("csv::../../tests/data/storages/csv/")
stor.get_exchanges()

['BINANCE.UM', 'HYPERLIQUID', 'KRAKEN']

In [3]:
stor.get_reader("BINANCE.UM", "SWAP").get_time_range("AAVEUSDT", "ohlc(1h)")

(numpy.datetime64('2023-06-01T00:00:00'),
 numpy.datetime64('2023-08-01T00:00:00'))

In [4]:
stor.get_reader("BINANCE.UM", "SWAP").get_time_range("BTCUSDT", "quote")

(numpy.datetime64('2017-08-24T13:01:12'),
 numpy.datetime64('2017-08-24T13:01:59'))

In [5]:
stor.get_reader("KRAKEN", "SWAP").get_time_range("AAVEUSD", "ohlc(1h)")

(numpy.datetime64('2023-06-01T00:00:00'),
 numpy.datetime64('2023-08-01T00:00:00'))

In [6]:
stor.get_reader("KRAKEN", "SWAP").read("BTCUSD", "quote").to_records()[:3]

[[2025-12-01T00:00:00.000000000]	90388.00000 (0.01) | 90389.00000 (0.01),
 [2025-12-01T00:00:01.000000000]	90380.00000 (0.01) | 90386.00000 (0.01),
 [2025-12-01T00:00:02.000000000]	90364.00000 (0.01) | 90370.00000 (0.01)]

In [7]:
stor.get_reader("KRAKEN", "SWAP").read("BTCUSD", "quote").transform(TypedGenericSeries())

                         bid      ask  bid_size  ask_size
time                                                     
2025-12-01 00:00:00  90388.0  90389.0    0.0101    0.0113
2025-12-01 00:00:01  90380.0  90386.0    0.0101    0.0113
2025-12-01 00:00:02  90364.0  90370.0    0.0109    0.0113
2025-12-01 00:00:03  90366.0  90367.0    0.0050    0.0060
2025-12-01 00:00:04  90358.0  90361.0    0.0005    0.0236
...                      ...      ...       ...       ...
2025-12-01 23:59:55  86317.0  86318.0    0.0022    0.0071
2025-12-01 23:59:56  86317.0  86318.0    0.0022    0.0131
2025-12-01 23:59:57  86317.0  86318.0    0.0022    0.1869
2025-12-01 23:59:58  86291.0  86298.0    0.0641    0.0391
2025-12-01 23:59:59  86290.0  86291.0    0.0050    0.1792

[83631 rows x 4 columns]

In [11]:
_X1 = pd.DataFrame.from_dict({
    "2025-01-01 00:00": {"open": 1000, "high": 1100, "low": 950, "close": 1050 },
    "2025-01-01 01:00": {"open": 1050, "high": 1200, "low": 900, "close": 1075 },
    "2025-01-01 02:00": {"open": 1075, "high": 1300, "low": 1050, "close": 1100 },
}, orient="index")
_X1.index = pd.DatetimeIndex(_X1.index)

_X2 = pd.DataFrame.from_dict({
    "2025-01-01 00:00": {"open": 100, "high": 110, "low": 95, "close": 105 },
    "2025-01-01 01:00": {"open": 105, "high": 120, "low": 90, "close": 107 },
    "2025-01-01 02:00": {"open": 107, "high": 130, "low": 105, "close": 110 },
}, orient="index")
_X2.index = pd.DatetimeIndex(_X2.index)

In [12]:
stor = StorageRegistry.get("handy", data={
    "BINANCE.UM:SWAP:BTCUSDT": _X1,
    "BINANCE.UM:SWAP:ETHUSDT": _X2
})
r = stor["BINANCE.UM", "SWAP"]

In [13]:
r.read(["BTCUSDT", "ETHUSDT"], "ohlc(1h)")

-[MULTI DATA]-
 | BTCUSDT(ohlc(1h))[2025-01-01 00:00:00 : 2025-01-01 02:00:00] : (3 x 5)
 | ETHUSDT(ohlc(1h))[2025-01-01 00:00:00 : 2025-01-01 02:00:00] : (3 x 5)

In [14]:
class Test1(IStrategy):
    def on_init(self, initializer: IStrategyInitializer):
        pass

    def on_market_data(self, ctx: IStrategyContext, data: MarketEvent) -> list[Signal] | Signal | None:
        pass

In [30]:
DataType.from_str("ohlc_quotes(1h)")

(ohlc_quotes, {'timeframe': '1h'})

In [36]:
handle_start_stop("-1d", None)

ValueError: First argument is delta but stop time is not defined !

In [32]:
simulate(
    Test1(), 
    { "ohlc": stor, "quote": stor, "some": stor}, 
    # stor,
    capital=1000, start="2020-01-01",
    instruments=[
        "BINANCE.UM:SWAP:BTCUSDT",
        "HYPERLIQUID:SWAP:BTCUSDC"
    ], 
    debug="DEBUG"
)

2026-02-05 12:10:09.607 [ 🐞 ] (runner) [Simulator] :: Preparing simulated trading on ['BINANCE.UM', 'HYPERLIQUID'] for 1000 USDT...
2026-02-05 12:10:09.607 [ ❌ ] (simulator) Simulation setup 0 failed with error: 'SimulationDataConfig' object has no attribute 'data_providers'


[]

2026-02-05 12:10:09.610 [ ❌ ] (simulator) Traceback (most recent call last):
  File "/home/quant0/devs/Qubx/src/qubx/backtester/simulator.py", line 274, in _run_setup
    portfolio_log_freq=portfolio_log_freq,
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/quant0/devs/Qubx/src/qubx/backtester/runner.py", line 130, in __init__
    self.ctx = self._create_backtest_context()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/quant0/devs/Qubx/src/qubx/backtester/runner.py", line 407, in _create_backtest_context
    self.data_config.data_providers,
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'SimulationDataConfig' object has no attribute 'data_providers'. Did you mean: 'data_provider'?

2026-02-05 12:10:09.610 [ ⚠️ ] (simulator) qubx.backtester.simulator:_run_setups:229 - Simulation setup 0 failed - skipping from results
2026-02-05 12:10:09.610 [ ⚠️ ] (simulator) qubx.backtester.simulator:_run_setups:229 - Simulation setup 0 failed - skipping from results


In [33]:
xs = StorageRegistry.get("qdb::quantlab")

In [34]:
rs = xs.get_reader("HYPERLIQUID", "SWAP")
rb = xs.get_reader("BINANCE.UM", "SWAP")

In [36]:
rs.get_data_types("BTCUSDC")

[funding_payment,
 orderbook,
 'ohlc(1min)',
 'ohlc(2min)',
 'ohlc(3min)',
 'ohlc(5min)',
 'ohlc(10min)',
 'ohlc(15min)',
 'ohlc(30min)',
 'ohlc(1h)',
 'ohlc(2h)',
 'ohlc(3h)',
 'ohlc(4h)',
 'ohlc(6h)',
 'ohlc(8h)',
 'ohlc(12h)',
 'ohlc(1d)',
 'ohlc(1w)',
 open_interest,
 quote,
 funding_rate]

In [48]:
for x in rs.read(["BTCUSDC", "AAVEUSDC", "ETHUSDC", "LTCUSDC"], "ohlc(1h)", "2025-01-01", "2026-01-01 00:10"):
    # x.to_pd().to_csv(f"../../tests/data/storages/csv/HYPERLIQUID/SWAP/{x.data_id}.1H.csv.gz")
    print(x)

LTCUSDC(ohlc(1h))[2025-01-01 00:00:00 : 2026-01-01 00:00:00] : (8761 x 11)
ETHUSDC(ohlc(1h))[2025-01-01 00:00:00 : 2026-01-01 00:00:00] : (8761 x 11)
BTCUSDC(ohlc(1h))[2025-01-01 00:00:00 : 2026-01-01 00:00:00] : (8761 x 11)
AAVEUSDC(ohlc(1h))[2025-01-01 00:00:00 : 2026-01-01 00:00:00] : (8761 x 11)


In [53]:
xx1 = rs.read(["BTCUSDC", "AAVEUSDC", "ETHUSDC", "LTCUSDC"], "ohlc(1h)", "2025-01-01", "2026-01-01 00:10")

In [ ]:
xx1.to_pd(1).head()